In [4]:
# ============================================================
# AGGRESSIVE PIPELINE - Target 93% Macro F1
# ============================================================
import os
os.environ['PYTHONWARNINGS'] = 'ignore'
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

In [5]:
# ============================================================
# STEP 1: LOAD DATA
# ============================================================
test  = pd.read_csv('/Users/damacm1124/Desktop/FHI/Financial-Health-Prediciton/Test.csv')
train = pd.read_csv('/Users/damacm1124/Desktop/FHI/Financial-Health-Prediciton/Train.csv')
print(f"Train shape: {train.shape} | Test shape: {test.shape}")
print(f"\nTarget:\n{train['Target'].value_counts()}")
print(f"\nHigh per country:\n{train[train['Target']=='High']['country'].value_counts()}")


Train shape: (9618, 39) | Test shape: (2405, 38)

Target:
Target
Low       6280
Medium    2868
High       470
Name: count, dtype: int64

High per country:
country
eswatini    307
malawi       96
zimbabwe     61
lesotho       6
Name: count, dtype: int64


In [6]:
# ============================================================
# STEP 2: PREPROCESSING
# ============================================================
for df in [train, test]:
    df['country'] = df['country'].str.strip().str.title()

test_ids   = test['ID'].copy()
y_raw      = train['Target'].copy()
train_feat = train.drop(columns=['ID', 'Target']).copy()
test_feat  = test.drop(columns=['ID']).copy()

In [7]:
# ============================================================
# STEP 3: ACCESS FEATURES (before encoding while still strings)
# ============================================================
def add_access_features(df):
    df = df.copy()
    df['financial_access_score'] = (
        (df['has_credit_card']      == 'Have now').astype(int) +
        (df['has_debit_card']       == 'Have now').astype(int) +
        (df['has_internet_banking'] == 'Have now').astype(int) +
        (df['has_loan_account']     == 'Have now').astype(int) +
        (df['has_mobile_money']     == 'Have now').astype(int)
    )
    df['insurance_score'] = (
        (df['has_insurance']           == 'Have now').astype(int) +
        (df['medical_insurance']       == 'Have now').astype(int) +
        (df['funeral_insurance']       == 'Have now').astype(int) +
        (df['motor_vehicle_insurance'] == 'Have now').astype(int)
    )
    df['total_financial_score'] = df['financial_access_score'] + df['insurance_score']
    df['has_any_insurance']     = (df['insurance_score'] > 0).astype(int)
    df['has_any_banking']       = (df['financial_access_score'] > 0).astype(int)
    df['positive_attitude']     = (
        (df['attitude_stable_business_environment'] == 'Yes') &
        (df['attitude_more_successful_next_year']   == 'Yes')
    ).astype(int)
    df['worried_business']      = (df['attitude_worried_shutdown'] == 'Yes').astype(int)
    df['uses_informal_finance'] = (
        (df['uses_informal_lender']       == 'Have now') |
        (df['uses_friends_family_savings']== 'Have now')
    ).astype(int)
    df['keeps_records_flag']    = (df['keeps_financial_records'] == 'Yes').astype(int)
    df['offers_credit_flag']    = (df['offers_credit_to_customers'] == 'Yes').astype(int)
    return df

train_feat = add_access_features(train_feat)
test_feat  = add_access_features(test_feat)

print(f"\nfinancial_access_score dist:\n{train_feat['financial_access_score'].value_counts()}")
print(f"insurance_score dist:\n{train_feat['insurance_score'].value_counts()}")




financial_access_score dist:
financial_access_score
0    4309
1    3730
2    1028
3     412
4     136
5       3
Name: count, dtype: int64
insurance_score dist:
insurance_score
0    7809
1    1396
2     326
3      87
Name: count, dtype: int64


In [8]:
# ============================================================
# STEP 4: NUMERIC FEATURE ENGINEERING
# ============================================================
def engineer_features(df):
    df = df.copy()
    df['profit_proxy']       = df['business_turnover'] - df['business_expenses']
    df['expense_ratio']      = df['business_expenses']  / (df['business_turnover'] + 1)
    df['income_vs_expense']  = df['personal_income']    / (df['business_expenses'] + 1)
    df['turnover_per_age']   = df['business_turnover']  / (df['owner_age'] + 1)
    df['income_per_age']     = df['personal_income']    / (df['owner_age'] + 1)
    df['turnover_vs_income'] = df['business_turnover']  / (df['personal_income'] + 1)
    df['turnover_per_year']  = df['business_turnover']  / (df['business_age_years'] + 1)
    df['total_age_months']   = (df['business_age_years'].fillna(0) * 12 +
                                df['business_age_months'].fillna(0))
    df['is_new_business']    = (df['business_age_years'] <= 1).astype(int)
    df['is_mature_business'] = (df['business_age_years'] >= 5).astype(int)
    for col in ['personal_income','business_expenses','business_turnover','profit_proxy']:
        df[f'log_{col}'] = np.log1p(df[col].clip(lower=0))
    for col in ['personal_income','business_turnover']:
        df[f'sqrt_{col}'] = np.sqrt(df[col].clip(lower=0))
    return df

train_feat = engineer_features(train_feat)
test_feat  = engineer_features(test_feat)

In [9]:
# ============================================================
# STEP 5: IMPUTATION
# ============================================================
num_cols = train_feat.select_dtypes(include='number').columns.tolist()
cat_cols = [c for c in train_feat.select_dtypes(include='object').columns if c != 'country']

for col in num_cols:
    train_feat[col] = train_feat.groupby(train_feat['country'])[col]\
                        .transform(lambda x: x.fillna(x.median()))
    train_feat[col] = train_feat[col].fillna(train_feat[col].median())
    test_feat[col]  = test_feat.groupby(test_feat['country'])[col]\
                        .transform(lambda x: x.fillna(x.median()))
    test_feat[col]  = test_feat[col].fillna(train_feat[col].median())

for col in cat_cols:
    for df in [train_feat, test_feat]:
        df[col] = df.groupby('country')[col].transform(
            lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else 'Unknown'))
        df[col] = df[col].fillna('Unknown')

print(f"\nMissing - Train: {train_feat.isnull().sum().sum()} | Test: {test_feat.isnull().sum().sum()}")



Missing - Train: 0 | Test: 0


In [10]:
# ============================================================
# STEP 6: ENCODE
# ============================================================
all_cat_cols = train_feat.select_dtypes(include='object').columns.tolist()
combined     = pd.concat([train_feat, test_feat], axis=0).reset_index(drop=True)

for col in all_cat_cols:
    le = LabelEncoder()
    combined[col] = le.fit_transform(combined[col].astype(str))

train_enc = combined.iloc[:len(train_feat)].reset_index(drop=True)
test_enc  = combined.iloc[len(train_feat):].reset_index(drop=True)

le_target = LabelEncoder()
y = le_target.fit_transform(y_raw.astype(str))
print(f"\nClasses: {le_target.classes_}")
print(f"Distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

X      = train_enc.values.astype(float)
X_test = test_enc.values.astype(float)
feature_cols = list(train_enc.columns)



Classes: ['High' 'Low' 'Medium']
Distribution: {np.int64(0): np.int64(470), np.int64(1): np.int64(6280), np.int64(2): np.int64(2868)}


In [11]:
# ============================================================
# LESOTHO SPECIAL TREATMENT
# ============================================================
# Lesotho has only 6 High samples — boost it with cross-country data
lesotho_mask    = train['country'] == 'Lesotho'
non_les_mask    = train['country'] != 'Lesotho'

X_lesotho       = X[lesotho_mask.values]
y_lesotho       = y[lesotho_mask.values]

# Add 30% of other countries High samples to Lesotho training
other_high_mask = (~lesotho_mask.values) & (y == 0)  # 0 = High
X_other_high    = X[other_high_mask]
y_other_high    = y[other_high_mask]

# Sample 30%
np.random.seed(42)
idx = np.random.choice(len(X_other_high), size=int(len(X_other_high)*0.3), replace=False)
X_les_boost = np.vstack([X_lesotho, X_other_high[idx]])
y_les_boost = np.concatenate([y_lesotho, y_other_high[idx]])

print(f"\nLesotho boosted training set: {len(y_les_boost)} samples")
print(f"High class in boosted set: {(y_les_boost==0).sum()}")


Lesotho boosted training set: 2083 samples
High class in boosted set: 145


In [ ]:
from xgboost import XGBClassifier
import lightgbm as lgb

countries = train['country'].unique()

def get_models():
    rf = RandomForestClassifier(
        n_estimators=1000, min_samples_leaf=1, min_samples_split=2,
        max_features='sqrt', class_weight='balanced_subsample',
        random_state=42, n_jobs=-1
    )
    xgb = XGBClassifier(
        n_estimators=1200,
        learning_rate=0.015,
        max_depth=7,
        min_child_weight=3,
        gamma=0.1,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.2,
        reg_lambda=1.5,
        eval_metric='mlogloss',
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    )
    lgbm = lgb.LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.015,
        max_depth=-1,
        num_leaves=127,
        min_child_samples=20,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=1.2,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    
    return rf, xgb, lgbm

# Add country to encoded data
train_enc['_country'] = train['country'].values
test_enc['_country']  = test['country'].values
feature_cols = [c for c in train_enc.columns if c != '_country']

# ── LESOTHO BOOST: add High samples from other countries ──
lesotho_mask   = train_enc['_country'] == 'Lesotho'
other_mask     = train_enc['_country'] != 'Lesotho'
other_high_mask= other_mask & (y == 0)  # High class from other countries

# Take 50% of other countries High samples to boost Lesotho
np.random.seed(42)
other_high_idx = np.where(other_high_mask)[0]
boost_idx      = np.random.choice(other_high_idx,
                   size=int(len(other_high_idx) * 0.5), replace=False)

X_lesotho_boost = np.vstack([
    train_enc.loc[lesotho_mask, feature_cols].values.astype(float),
    train_enc.iloc[boost_idx][feature_cols].values.astype(float)
])
y_lesotho_boost = np.concatenate([y[lesotho_mask], y[boost_idx]])

print(f"Lesotho boosted: {len(y_lesotho_boost)} samples")
print(f"High in boosted: {(y_lesotho_boost==0).sum()}")

print("\n" + "="*50)
print("PER-COUNTRY CROSS VALIDATION")
print("="*50)

for country in countries:
    if country == 'Lesotho':
        X_c = X_lesotho_boost
        y_c = y_lesotho_boost
    else:
        mask = train_enc['_country'] == country
        X_c  = train_enc.loc[mask, feature_cols].values.astype(float)
        y_c  = y[mask]

    print(f"\n{country}: {len(y_c)} samples | High={(y_c==0).sum()} Low={(y_c==1).sum()} Medium={(y_c==2).sum()}")

    min_class = np.bincount(y_c).min()
    n_splits  = min(5, min_class) if min_class >= 2 else 2

    if min_class < 2:
        print(f"  Skipping CV — too few samples")
        continue

    fold_scores = []
    skf_c = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for tr_idx, val_idx in skf_c.split(X_c, y_c):
        X_tr, X_val = X_c[tr_idx], X_c[val_idx]
        y_tr, y_val = y_c[tr_idx], y_c[val_idx]

        min_c = np.bincount(y_tr).min()
        if min_c >= 2:
            try:
                k  = min(3, min_c - 1)
                sm = SMOTE(random_state=42, k_neighbors=k)
                X_tr_sm, y_tr_sm = sm.fit_resample(X_tr, y_tr)
            except:
                X_tr_sm, y_tr_sm = X_tr, y_tr
        else:
            X_tr_sm, y_tr_sm = X_tr, y_tr

        rf, xgb, lgbm = get_models()
        rf.fit(X_tr_sm, y_tr_sm)
        xgb.fit(X_tr_sm, y_tr_sm)
        lgbm.fit(X_tr_sm, y_tr_sm)

        p = (rf.predict_proba(X_val)   * 0.20 +
             xgb.predict_proba(X_val)  * 0.45 +
             lgbm.predict_proba(X_val) * 0.35 )

        fold_scores.append(f1_score(y_val, np.argmax(p, axis=1), average='macro'))

    print(f"  CV F1: {np.mean(fold_scores):.4f}")

Lesotho boosted: 2176 samples
High in boosted: 238

PER-COUNTRY CROSS VALIDATION

Eswatini: 2674 samples | High=307 Low=1375 Medium=992


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from imblearn.combine import SMOTETomek

gb = GradientBoostingClassifier(
    n_estimators=800, learning_rate=0.02, max_depth=6,
    min_samples_leaf=3, subsample=0.8, max_features='sqrt', random_state=42
)

sm = SMOTETomek(random_state=42)

print("Training final models...")
X_sm, y_sm = sm.fit_resample(X, y)

rf.fit(X_sm, y_sm)
gb.fit(X_sm, y_sm)
xgb.fit(X_sm, y_sm)

p       = (rf.predict_proba(X_c)   * 0.20 +
           xgb.predict_proba(X_c)  * 0.45 +
           lgbm.predict_proba(X_c) * 0.35)
p = p ** 1.2
p = p / p.sum(axis=1, keepdims=True)
test_preds = le_target.inverse_transform(np.argmax(p, axis=1))
submission = pd.DataFrame({'ID': test_ids, 'Target': test_preds})
# submission.to_csv('submission.csv', index=False)
# print("\n✅ submission.csv saved!")
# print(submission['Target'].value_counts())
# print(submission.head())

In [ ]:
print("financial_access_score:\n", train_feat['financial_access_score'].value_counts())
print("insurance_score:\n", train_feat['insurance_score'].value_counts())
print("total_financial_score:\n", train_feat['total_financial_score'].value_counts())

In [ ]:
# ============================================================
# STEP 9: FINAL TRAIN
# ============================================================
print("Training final per-country models...")
country_models = {}

for country in countries:
    if country == 'Lesotho':
        X_c = X_lesotho_boost
        y_c = y_lesotho_boost
    else:
        mask = train_enc['_country'] == country
        X_c  = train_enc.loc[mask, feature_cols].values.astype(float)
        y_c  = y[mask]

    min_c = np.bincount(y_c).min()
    if min_c >= 2:
        try:
            k  = min(3, min_c - 1)
            sm = SMOTE(random_state=42, k_neighbors=k)
            X_sm, y_sm = sm.fit_resample(X_c, y_c)
        except:
            X_sm, y_sm = X_c, y_c
    else:
        X_sm, y_sm = X_c, y_c

    rf, xgb, lgbm = get_models()
    rf.fit(X_sm, y_sm)
    xgb.fit(X_sm, y_sm)
    lgbm.fit(X_sm, y_sm)

    country_models[country] = (rf, xgb, lgbm)
    print(f"  {country}: trained on {len(X_sm)} samples ✅")

# Confusion matrix on original train
train_preds_all = np.zeros(len(train_enc), dtype=int)
for country in countries:
    mask          = train_enc['_country'] == country
    X_c           = train_enc.loc[mask, feature_cols].values.astype(float)
    rf, xgb, lgbm = country_models[country]
    p              = (rf.predict_proba(X_c)   * 0.20 +
                     xgb.predict_proba(X_c)  * 0.45 +
                     lgbm.predict_proba(X_c) * 0.35)
    train_preds_all[mask.values] = np.argmax(p, axis=1)

print("\nClassification Report:")
print(classification_report(y, train_preds_all, target_names=le_target.classes_))

cm = confusion_matrix(y, train_preds_all)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_target.classes_, yticklabels=le_target.classes_)
plt.title('Confusion Matrix - Per Country + Lesotho Boost', fontsize=14)
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# STEP 10: FEATURE IMPORTANCE
# ============================================================
feat_imp = pd.DataFrame({'Feature': feature_cols, 'Importance': rf.feature_importances_})
feat_imp = feat_imp.sort_values('Importance', ascending=False).head(20)
plt.figure(figsize=(10, 8))
sns.barplot(data=feat_imp, x='Importance', y='Feature', palette='viridis')
plt.title('Top 20 Feature Importances', fontsize=14)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# STEP 11: PREDICT + SAVE
# ============================================================
print("Predicting test set per country...")
test_preds_all = np.empty(len(test_enc), dtype=object)

for country in countries:
    mask          = test_enc['_country'] == country
    if mask.sum() == 0:
        continue
    X_c           = test_enc.loc[mask, feature_cols].values.astype(float)
    rf, xgb, lgbm = country_models[country]

    p     = (rf.predict_proba(X_c)   * 0.30 +
             xgb.predict_proba(X_c)  * 0.40 +
             lgbm.predict_proba(X_c) * 0.30)
    preds = le_target.inverse_transform(np.argmax(p, axis=1))
    test_preds_all[mask.values] = preds
    print(f"  {country}: {mask.sum()} samples predicted ✅")

submission = pd.DataFrame({'ID': test_ids, 'Target': test_preds_all})
submission.to_csv('submission.csv', index=False)

print("\n" + "="*50)
print("✅ submission.csv saved!")
print("="*50)
print(submission['Target'].value_counts())
print(submission.head())